**Download datasets from: https://github.com/yastrebksv/TennisCourtDetector**

**Unzip it to project root folder**

**Install torch and torchvision, we are going to use torch to train our datasets**

In [2]:
%pip install torch torchvision
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.
  Using cached opencv_python-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl.metadata (20 kB)
Using cached opencv_python-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl (37.3 MB)
Note: you may need to restart the kernel to use updated packages.


**Create torch dataset class**

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import json
import cv2
import numpy as np

device = torch.device('mps' if torch.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# Define the dataset class
class KeypointsDataset(Dataset):
    # Constructor with image directory and data file
    def __init__(self, image_dir, data_file):
        # Keep image directory
        self.image_dir = image_dir
        # Load the data file into a dictionary with keys: 'id', 'metric', 'kps'
        with open(data_file, 'r') as f:
            self.data = json.load(f)
        # Define the image transforms
        self.transforms = transforms.Compose([
            # Transform to PIL image so that we can do image transforms
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            # The mean is the average pixel values for RGB channels in ImageNet
            # The std is the standard deviation of these pixel values in ImageNet.
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Get the item at index idx, which is a dictionary with keys: 'id', 'metric', 'kps'
        item = self.data[idx]
        # Read the image using OpenCV
        image = cv2.imread(f"{self.image_dir}/{item['id']}.png")
        # image.shape is (height, width, channels)
        h, w = image.shape[:2]
        # Convert the image from BGR to RGB format
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # Transform the image to a tensor and resize it to 224x224
        # The transforms are defined in the constructor
        image = self.transforms(image)
        # Convert the keypoints to a numpy array and flatten it
        # The keypoints are in the format [[x1, y1], [x2, y2], ..., [xn, yn]]
        # where n is the number of keypoints
        kps = np.array(item['kps']).flatten()
        # Convert the keypoints to a float32 numpy array
        kps = kps.astype(np.float32)
        # Scale the keypoints width to match the resized image size
        kps[::2] *= 224.0 / w
        # Scale the keypoints height to match the resized image size
        kps[1::2] *= 224.0 / h

        return image, kps

Using device: mps


**Create model**

In [5]:
# Define training dataset
train_dataset = KeypointsDataset('data/images', 'data/data_train.json')
# Define validation dataset
val_dataset = KeypointsDataset('data/images', 'data/data_val.json')

# Load the training datasets into DataLoader objects
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
# Load the validation datasets into DataLoader objects
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

# Load a pre-trained ResNet50 model
# Pre-trained models are trained on ImageNet dataset
# and can be used for transfer learning
# The model is a ResNet50 architecture with 50 layers
# The model outputs a 1000-dimensional vector
model = models.resnet50(pretrained=True)
# Replace the last layer of the above resnet model with a new one to predict 14 keypoints
# The model outputs a 14-dimensional vector (x, y) for each keypoint
# The last layer of the ResNet50 model is a fully connected layer with 1000 outputs
# We replace it with a new fully connected layer with 14 * 2 outputs (x, y) for each keypoint
model.fc = torch.nn.Linear(model.fc.in_features, 14 * 2) 
# Move the model to the device (GPU or CPU or MPS)
model = model.to(device)

/Users/jasontan/dev/workspace/tennis-scorer/tennis-4/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/jasontan/dev/workspace/tennis-scorer/tennis-4/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


**Train the model**

In [6]:
# Define the loss function and optimizer
# The loss function is Mean Squared Error (MSE) loss
criterion = torch.nn.MSELoss()
# The optimizer is Adam with a learning rate of 0.0001
# The learning rate is the step size for the optimizer
# The optimizer updates the model parameters to minimize the loss function
# The loss function measures how well the model is doing
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
epochs = 20
for epoch in range(epochs):
    for i, (image, kps) in enumerate(train_loader):
        image = image.to(device)
        kps = kps.to(device)
        # Zero the gradients before the backward pass
        # The gradients are accumulated by default in PyTorch
        # We need to zero them before the backward pass
        # The backward pass computes the gradients of the loss function with respect to the model parameters
        # The optimizer updates the model parameters using these gradients
        optimizer.zero_grad()
        # Forward pass: compute the model output
        # The model takes the image as input and outputs a 14-dimensional vector (x, y) for each keypoint
        # The output is a tensor of shape (batch_size, 14 * 2)
        # The keypoints are in the format [[x1, y1], [x2, y2], ..., [xn, yn]]
        # where n is the number of keypoints
        outputs = model(image)
        # Compute the loss between the model output and the ground truth keypoints
        # The loss is a measure of how well the model is doing
        loss = criterion(outputs, kps)
        # Backward pass: compute the gradients of the loss function with respect to the model parameters
        loss.backward()
        # Update the model parameters using the optimizer
        # The optimizer updates the model parameters to minimize the loss function
        # The optimizer uses the gradients computed in the backward pass
        optimizer.step()
        
        if i % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Step {i+1}/{len(train_loader)}, Loss: {loss.item()}")

Epoch 1/20, Step 1/829, Loss: 14435.306640625
Epoch 1/20, Step 11/829, Loss: 14787.9296875
Epoch 1/20, Step 21/829, Loss: 14758.654296875
Epoch 1/20, Step 31/829, Loss: 13892.7646484375
Epoch 1/20, Step 41/829, Loss: 13640.4287109375
Epoch 1/20, Step 51/829, Loss: 13440.841796875
Epoch 1/20, Step 61/829, Loss: 12631.384765625
Epoch 1/20, Step 71/829, Loss: 11723.8173828125
Epoch 1/20, Step 81/829, Loss: 12158.947265625
Epoch 1/20, Step 91/829, Loss: 10774.8544921875
Epoch 1/20, Step 101/829, Loss: 12091.728515625
Epoch 1/20, Step 111/829, Loss: 10607.2255859375
Epoch 1/20, Step 121/829, Loss: 10605.75
Epoch 1/20, Step 131/829, Loss: 10594.2138671875
Epoch 1/20, Step 141/829, Loss: 9459.123046875
Epoch 1/20, Step 151/829, Loss: 9272.0830078125
Epoch 1/20, Step 161/829, Loss: 8592.572265625
Epoch 1/20, Step 171/829, Loss: 8885.8701171875
Epoch 1/20, Step 181/829, Loss: 8355.8388671875
Epoch 1/20, Step 191/829, Loss: 8182.89111328125
Epoch 1/20, Step 201/829, Loss: 8053.50048828125
Epoch 

**Save the trained model**

In [7]:
torch.save(model.state_dict(), 'keypoints_model_resnet50_epoch20_mps.pth')